# 01 — Scoring output review

After running `python -m src.pipeline.run_batch_scoring` (from the repo root), use this notebook to:

- Check **schema** and **row alignment** with the input batch  
- Validate **probabilities** (0–1, no NaNs)  
- Inspect **distribution** of `churn_score` and **predicted_label** mix  

Works whether Jupyter’s working directory is the **repo root** or the **`notebooks/`** folder.

In [ ]:
from pathlib import Path

import pandas as pd


def resolve_repo_root() -> Path:
    """Find project root (contains data/ and src/)."""
    cwd = Path.cwd().resolve()
    for base in (cwd, cwd.parent):
        if (base / "src" / "pipeline").is_dir() and (base / "data").is_dir():
            return base
    raise FileNotFoundError(
        "Could not find repo root. Open Jupyter from the repo root or from notebooks/."
    )


REPO = resolve_repo_root()
OUTPUT_PATH = REPO / "data" / "scored_outputs" / "scored_batch_001.csv"
INPUT_PATH = REPO / "data" / "input_batches" / "batch_001.csv"

print("Repo:", REPO)
print("Output:", OUTPUT_PATH)

In [ ]:
if not OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {OUTPUT_PATH.name}. Run: cd {REPO} && python -m src.pipeline.run_batch_scoring"
    )

df = pd.read_csv(OUTPUT_PATH)
print("Scored shape:", df.shape)
df.head(8)

## Schema & data quality

Expected columns match the pipeline contract; probabilities should lie in `[0, 1]` with no missing values.

In [ ]:
required = [
    "customer_id",
    "churn_score",
    "predicted_label",
    "model_version",
    "scoring_timestamp",
]
for col in required:
    assert col in df.columns, f"Missing column: {col}"

assert df["churn_score"].between(0, 1, inclusive="both").all(), "churn_score outside [0, 1]"
assert not df["churn_score"].isna().any(), "NaN in churn_score"
assert df["predicted_label"].isin([0, 1]).all(), "predicted_label must be 0 or 1"

if INPUT_PATH.exists():
    df_in = pd.read_csv(INPUT_PATH)
    assert len(df) == len(df_in), f"Row count mismatch: scored={len(df)} vs input={len(df_in)}"
    print(f"Row count matches input batch ({len(df)} rows).")
else:
    print("Input batch not found; skipped row-count check.")

print("Schema & quality checks OK.")

## Churn score distribution

Skew toward low scores is common if most customers are low-risk; a long upper tail often drives the churn capture rate at your chosen threshold.

In [ ]:
df["churn_score"].describe()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["churn_score"], bins=30, color="#f0a030", edgecolor="#1a1206", alpha=0.85)
ax.set_xlabel("churn_score")
ax.set_ylabel("count")
ax.set_title("Distribution of predicted churn probability")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## Predicted labels

Share of rows scored as churn (`1`) depends on the configured threshold (default often ~0.35 in this project).

In [ ]:
vc = df["predicted_label"].value_counts().sort_index()
display(vc)
pct_churn = 100 * vc.get(1, 0) / len(df)
print(f"Predicted churn rate: {pct_churn:.1f}%")

## Model version & timestamp

All rows in one run share the same `scoring_timestamp` and `model_version` — useful for auditing which artifact produced the file.

In [ ]:
print("model_version:", df["model_version"].unique().tolist())
print("scoring_timestamp:", df["scoring_timestamp"].iloc[0])